# Continuous RUL Benchmark (CRB) — Google Colab (T4 GPU)

Runs entirely from GitHub — no Google Drive or other services.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**

Clone the repo → install → 2-minute smoke test → benchmark → leaderboard → results zip download.

Results are stored on the Colab machine while it runs (`ml_pipeline/results/benchmark/`).
If the page disconnects but the runtime is still alive, **Run all** again: finished runs are
skipped and it resumes. If the runtime is reset, results are lost — run in smaller chunks
with `DATASETS` / `MODELS` below and download the zip after each chunk.

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/Nafis878/Continuous-.git"
BRANCH   = "main"
WORKDIR  = "/content/Continuous"

SEEDS    = "42 43 44"
DATASETS = ""      # "" = all (FD001 FD002 FD003 FD004 PHM08); or a chunk, e.g. "FD001 FD003"
MODELS   = ""      # "" = all 12 models; or e.g. "CATT+TCT CATT TCN Transformer LSTM HistGB MeanRUL"
RUN_SMOKE_TEST   = True
RUN_SCANIA_STUDY = False   # separate classification study; downloads ~1.7 GB of Git LFS data from GitHub

## 1. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU (CPU works but is much slower).")

## 2. Clone / update the repo from GitHub

In [ ]:
import os
# Private repo? Uncomment to enter a GitHub token when the cell runs (never save it in the notebook):
# from getpass import getpass
# REPO_URL = f"https://{getpass('GitHub token: ')}@github.com/Nafis878/Continuous-.git"

os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"   # large LFS files are only fetched for the optional SCANIA study
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    # keeps untracked results (ml_pipeline/results/benchmark) so the run can resume
    !git -C {WORKDIR} fetch -q origin {BRANCH} && git -C {WORKDIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1
assert os.path.isfile("ml_pipeline/run_benchmark.py"), "clone failed - check REPO_URL / access"

## 3. Install dependencies

In [ ]:
!pip install -q -r ml_pipeline/requirements.txt
import numpy, pandas, sklearn, h5py
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__, "| h5py", h5py.__version__)
OUT = f"{WORKDIR}/ml_pipeline/results/benchmark"
print("Results directory:", OUT)

## 4. Smoke test (~2 min): every model, 1 epoch, small subsets

In [ ]:
if RUN_SMOKE_TEST:
    !python ml_pipeline/run_benchmark.py --quick --fresh --out /content/crb_smoke
    import pandas as pd
    smoke = pd.read_csv("/content/crb_smoke/runs.csv")
    bad = smoke[smoke["status"] != "ok"]
    assert bad.empty, f"smoke test failures:\n{bad[['dataset','model','status']]}"
    print(f"Smoke test OK: {len(smoke)} runs")

## 5. Benchmark

Every finished run is appended to `runs.csv` immediately and the leaderboard is rebuilt.
Re-running this cell (same runtime) skips finished runs.

In [ ]:
args = f'--seeds {SEEDS} --out "{OUT}"'
if DATASETS.strip():
    args += f" --datasets {DATASETS}"
if MODELS.strip():
    args += f" --models {MODELS}"
print("python ml_pipeline/run_benchmark.py", args)
!python ml_pipeline/run_benchmark.py {args}

## 6. Leaderboard

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
lb = os.path.join(OUT, "LEADERBOARD.md")
if os.path.exists(lb):
    display(Markdown(open(lb, encoding="utf-8").read()))
    display(pd.read_csv(os.path.join(OUT, "leaderboard_avg_rank.csv")))
else:
    print("No leaderboard yet.")

## 7. Download results (zip)

In [ ]:
import shutil, time
archive = shutil.make_archive(f"/content/crb-results-{time.strftime('%Y%m%d-%H%M%S')}", "zip", OUT)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("Automatic download unavailable:", exc, "-> use the Files panel (left sidebar) to download it.")

## 8. Optional: SCANIA classification study + classic RUL scripts

Not part of CRB. Downloads the SCANIA data (~1.7 GB) from GitHub LFS.

In [ ]:
if RUN_SCANIA_STUDY:
    !git lfs install --skip-repo
    !git lfs pull --include "SCANIA-Component-X/data/*"
    !python ml_pipeline/run_family_comparison.py --seeds {SEEDS}
    !python ml_pipeline/run_scania_cost_tune.py --seeds {SEEDS}
    !python ml_pipeline/run_condition_rul.py --seeds {SEEDS}
    archive2 = shutil.make_archive("/content/family-comparison-results", "zip", "ml_pipeline/results/family_comparison")
    try:
        files.download(archive2)
    except Exception as exc:
        print("download via Files panel:", archive2, exc)